In [55]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os


In [56]:
embedings_cache_path = "./outputs/output_embeddings_small/embeddings_cache_keyed.pkl"
#load from pickle
import pickle
with open(embedings_cache_path, "rb") as f:
    embeddings_cache = pickle.load(f)

#loaded embeddings cache is a dict of text to embedding
print(f"Loaded embeddings cache with {len(embeddings_cache)} entries")

human_embedings = 0
vlm_embedings = 0
for key in embeddings_cache.keys():
    #key is a tuple of agent video question answer
    if "human" in key[0]:
        human_embedings += 1
        #print(f"Human embedding key: {key}")
    else:
        vlm_embedings += 1
        #print(f"VLM embedding key: {key}")
        
        
print(f"Human embedings: {human_embedings}")
print(f"VLM embedings: {vlm_embedings}")

Loaded embeddings cache with 10000 entries
Human embedings: 6000
VLM embedings: 4000


In [57]:
#load answers text cache
answers_text_cache_path = "./data/r2.csv"
df_answers = pd.read_csv(answers_text_cache_path)
#remove duplicates
df_answers = df_answers.drop_duplicates(subset=["AGENT", "VIDEO", "QUESTION_NUM"])
#divide by blocks
def get_block(question_num):
    if question_num <= 5:
        return 1
    elif question_num <= 10:
        return 2
    elif question_num <= 15:
        return 3
    elif question_num <= 20:
        return 4
df_answers["BLOCK"] = df_answers["QUESTION_NUM"].apply(get_block)
print(f"Loaded answers text cache with {len(df_answers)} entries")
print(df_answers.head())

Loaded answers text cache with 10000 entries
          AGENT         VIDEO  BLOCK  QUESTION_NUM  REPETITION  \
0  human_lima_1  Robusto2_153      1             1           1   
1  human_lima_2  Robusto2_153      1             1           1   
2  human_lima_3  Robusto2_153      1             1           1   
3  human_lima_4  Robusto2_153      1             1           1   
4  human_lima_5  Robusto2_153      1             1           1   

                                              ANSWER  
0  The ego vehicle is accelerating slowly because...  
1            The ego vehicle is turning to the right  
2  the ego vehicle brakes and steers slightly to ...  
3                                   Braking to yield  
4  The ego vehicle is moving forward while mainta...  


In [58]:
#first reduce all embedding cache using pca to 2 dimensions and store then into a df with agent video question answer and embedding
from sklearn.decomposition import PCA
from umap import UMAP
embeddings = []
for key, embedding in embeddings_cache.items():
    embeddings.append(embedding)

# --- Crear dataframe base ---
keys = list(embeddings_cache.keys())

idx = pd.MultiIndex.from_tuples(keys, names=["AGENT", "VIDEO", "QUESTION_NUM"])
blocks = df_answers.set_index(["AGENT", "VIDEO", "QUESTION_NUM"])["BLOCK"].reindex(idx).to_numpy()

base = pd.DataFrame({
    "AGENT": [k[0] for k in keys],
    "VIDEO": [k[1] for k in keys],
    "QUESTION_NUM": [k[2] for k in keys],
    "BLOCK": blocks,
})

embeddings_arr = np.vstack(embeddings)
coords_PCA = np.zeros((len(keys), 2))
coords_UMAP = np.zeros((len(keys), 2))
for block in [1, 2, 3, 4]:
    mask = base["BLOCK"] == block
    print(f"Processing block {block} with {mask.sum()} entries")
    if mask.any():
        pca_block = PCA(n_components=2)
        umap_block = UMAP(n_components=2,metric="cosine",local_connectivity = 1, n_jobs=16, random_state=42)
        coords_PCA[mask.values] = pca_block.fit_transform(embeddings_arr[mask.values])
        coords_UMAP[mask.values] = umap_block.fit_transform(embeddings_arr[mask.values])

interactive = base.assign(pca_X=coords_PCA[:, 0], pca_Y=coords_PCA[:, 1])
interactive = interactive.assign(umap_X=coords_UMAP[:, 0], umap_Y=coords_UMAP[:, 1])

# --- Merge vectorizado ---
interactive = interactive.merge(
    df_answers[["AGENT", "VIDEO", "QUESTION_NUM", "ANSWER"]],
    on=["AGENT", "VIDEO", "QUESTION_NUM"],
    how="left"
)

def get_group_color(agent):
    if "human" in agent:
        if "nyc" in agent:
            return "nyc"
        else:
            return "lima"
    else:
        return "vlm"
    
color_map = {
    "nyc": "#0000ff",
    "lima": "#ff0000",
    "vlm": "#00ff00",
}
    
    

interactive["group"] = interactive["AGENT"].map(get_group_color)
print(interactive.head())


Processing block 1 with 2500 entries


/home/andre/miniconda3/envs/torch/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Processing block 2 with 2500 entries


/home/andre/miniconda3/envs/torch/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Processing block 3 with 2500 entries


/home/andre/miniconda3/envs/torch/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Processing block 4 with 2500 entries


/home/andre/miniconda3/envs/torch/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


          AGENT         VIDEO  QUESTION_NUM  BLOCK     pca_X     pca_Y  \
0  human_lima_1  Robusto2_153             1      1  0.421279  0.072409   
1  human_lima_2  Robusto2_153             1      1  0.518072  0.042419   
2  human_lima_3  Robusto2_153             1      1  0.455936 -0.004245   
3  human_lima_4  Robusto2_153             1      1  0.043912  0.019573   
4  human_lima_5  Robusto2_153             1      1  0.512219 -0.101658   

      umap_X     umap_Y                                             ANSWER  \
0  15.851650   9.897373  The ego vehicle is accelerating slowly because...   
1  15.805429  13.544082            The ego vehicle is turning to the right   
2  18.779915  10.061336  the ego vehicle brakes and steers slightly to ...   
3  18.133673   6.579480                                   Braking to yield   
4  15.318124  10.963581  The ego vehicle is moving forward while mainta...   

  group  
0  lima  
1  lima  
2  lima  
3  lima  
4  lima  


In [61]:
import jscatter
import pandas as pd
import numpy as np

BLOCK_TO_PLOT = int(input("Enter block number to plot (1-4): "))

# Sample data
# Create an interactive scatter plot
scatter = jscatter.Scatter(
    data=interactive[interactive["BLOCK"] == BLOCK_TO_PLOT],  # Filter for block 1
    x='umap_X',
    y='umap_Y',
    color_by='group',
    color_map=color_map,
    opacity=0.7,
    width=800,
    height=600,   
)
scatter.axes(grid=True)
scatter.axes(labels=['PCA 1', 'PCA 2'])
scatter.tooltip(
  enable=True,
  size="medium",
  properties=["AGENT", "VIDEO", "QUESTION_NUM", "ANSWER"],
  
)

scatter.show()